#### 1. Quantization/export improvements

This is the highest-ROI path on the official leaderboard.

Why:

- The current official leader (as of apr1) improved from `1.1194` to `1.1147` mainly through quantization/export changes and zero-parameter compute allocation changes.
- Export-time gains are cheap relative to training-time architecture rewrites.

If we compare:

  - baseline: running root train_gpt.py with its default env vars
  - experiment: running colab/2026-04-06_QuantExport_Improvements_Benchmark/run.sh

  then the 2026-04-06 launcher changes two different things:

  1. benchmark/training setup
  2. export behavior

  Baseline root defaults in train_gpt.py:

  - trains on the normal fineweb10B_sp1024 path directly
  - uses all train shards visible in that dataset path
  - TRAIN_BATCH_TOKENS=524288
  - TRAIN_SEQ_LEN=1024
  - VAL_BATCH_SIZE=524288
  - WARMDOWN_ITERS=1200
  - VAL_LOSS_EVERY=1000
  - SEED=1337
  - ENABLE_COMPILE=1
  - ENABLE_FUSED_ADAM=1
  - export format defaults to:
      - int8 quantization
      - zlib compression
      - clip percentile 99.99984
      - small tensors kept as float up to 65536 elements
      - kept-float storage dtype fp16
      - per-row scale dtype fp16

  The 2026-04-06 Colab launcher changes all of these:

  - creates a local 10-shard dataset view for comparability
  - forces TRAIN_SHARDS=10
  - reduces train/val batch to 65536
  - increases TRAIN_SEQ_LEN to 2048
  - sets WARMDOWN_ITERS=4000
  - sets VAL_LOSS_EVERY=4000
  - sets SEED=314
  - disables compile by default
  - disables fused Adam by default
  - enables math SDP fallback now for Colab stability
  - changes export compression from zlib to lzma

  So if you measure a different result from run.sh, that result is a mix of:

  - different training setup
  - different runtime behavior
  - different export compression

  The root trainer exports by:

  1. taking trained weights
  2. quantizing most float tensors to int8
  3. storing some tensors in float instead of int8
  4. serializing that quantized object with torch.save
  5. compressing the serialized bytes with a general-purpose compressor

  That logic lives in train_gpt.py.

  there is:

  - no architecture change
  - no optimizer change
  - no retraining trick
  - only changing how the final artifact is quantized/stored/compressed

  That is cheap in the sense that it happens after training.

---


  INT8_CLIP_PERCENTILE

  - Before quantizing a float tensor to int8, the code clips very large outlier values.
  - It uses the absolute-value percentile as the clipping threshold.
  - Example: 99.99984 means “ignore only the most extreme outliers”.
  - Why this helps:
      - if you do not clip, a few huge outliers can waste most of the int8 range
      - if you clip too aggressively, you distort too many weights
  - So this is a quantization tradeoff between outlier handling and accuracy.

  INT8_KEEP_FLOAT_MAX_NUMEL

  - Tensors with at most this many elements are not quantized to int8.
  - They are kept in float form instead.
  - Why:
      - very small tensors often do not compress well with int8+metadata
      - quantizing them can hurt quality more than it helps size
  - So this threshold says which tensors are “too small to bother quantizing”.

  INT8_KEEP_FLOAT_STORE_DTYPE

  - When a tensor is kept as float, this says what float dtype to store it in.
  - fp16 is smaller than fp32.
  - So this saves bytes on those kept-float tensors.

  INT8_PER_ROW_SCALE_DTYPE

  - For 2D tensors, quantization is per-row.
  - Each row needs a saved scale.
  - This variable controls what dtype those scales use.
  - Smaller dtype means fewer bytes, but possibly noisier dequantization.

  Important subtle point
  In the current 2026-04-06 launcher, the INT8* values are explicitly set, but they are the same as the root defaults.

  So the actual export difference versus the root default run is mostly:

  - EXPORT_COMPRESSOR=lzma instead of zlib

  The INT8* variables are exposed and pinned for experimentation, but they are not yet changed away from the baseline defaults.

  So the accurate summary of what happens here is:

  - benchmark-aligned training/runtime changes
  - plus one real export change by default: lzma compression
  - plus exposed int8 quantization knobs that are ready to tune, but currently match the root defaults

In [ ]:
!git clone https://github.com/IanniMuliterno/parameter-golf.git
%cd parameter-golf/colab/2026-04-06_QuantExport_Improvements_Benchmark

Cloning into 'parameter-golf'...
remote: Enumerating objects: 636, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 636 (delta 5), reused 13 (delta 5), pack-reused 622 (from 2)
Receiving objects: 100% (636/636), 1.40 MiB | 16.86 MiB/s, done.
Resolving deltas: 100% (270/270), done.
/content/parameter-golf/colab/2026-04-06_QuantExport_Improvements_Benchmark


In [ ]:
!cd /content/parameter-golf && git pull

remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 12 (delta 4), reused 12 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (12/12), 4.45 KiB | 1.11 MiB/s, done.
From https://github.com/IanniMuliterno/parameter-golf
   6a95007..800998f  main       -> origin/main
Updating 6a95007..800998f
Fast-forward
 .../README.md                                      | 133 +++++++++++++++++++++
 .../requirements.txt                               |   2 +
 .../run.sh                                         |  90 ++++++++++++++
 .../train_gpt.py                                   |  12 ++
 4 files changed, 237 insertions(+)
 create mode 100644 colab/2026-04-06_QuantExport_Improvements_Benchmark/README.md
 create mode 100644 colab/2026-04-06_QuantExport_Improvements_Benchmark/requirements.txt
 create mode 100755 colab/2026-04-06_QuantExport_Improvements_Benchmark/run.sh
 create mode 100644 colab/2026-04-

In [ ]:
!INSTALL_DEPS=1 ENABLE_MATH_SDP=1 VAL_LOSS_EVERY=0 TRAIN_LOG_EVERY=1 ITERATIONS=300 bash run.sh

logs/2026-04-06_quant_export_improvements.txt
val_bpb:enabled tokenizer_kind=sentencepiece tokenizer_path=/content/parameter-golf/data/tokenizers/fineweb_1024_bpe.model
train_loader:dataset:fineweb10B_sp1024_10shards train_shards:10
val_loader:shards pattern=/content/parameter-golf/colab/2026-04-06_QuantExport_Improvements_Benchmark/runtime_data/fineweb10B_sp1024_10shards/fineweb_val_*.bin tokens:62021632
model_params:17059912
world_size:1 grad_accum_steps:8
compute_dtype:torch.bfloat16 muon_dtype:torch.bfloat16 compile:False fused_adam:False tf32:True
sdp_backends:cudnn=False flash:True mem_efficient:False math:True
attention_mode:gqa num_heads:8 num_kv_heads:4
tie_embeddings:True embed_lr:0.05 head_lr:0.0 matrix_lr:0.04 scalar_lr:0.04
train_batch_tokens:65536 train_seq_len:2048 iterations:300 warmup_steps:20 max_wallclock_seconds:600.000
seed:314
warmup_step:1/20
warmup_step:2/20
warmup_step:3/20
warmup_step:4/20
warmup_step:5/20
warmup_step:6/20
warmup_step:7/20
warmup_step:8/20
war